In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy(
    "user_id", "product_id", "event_time"
).orderBy(col("event_time").desc())

dedup_updates = (
    spark.table("ecommerce.events_table")
         .withColumn("rn", row_number().over(window_spec))
         .filter(col("rn") == 1)
         .drop("rn")
)


In [0]:
#incremental merge 

from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "ecommerce.events_delta")

deltaTable.alias("t").merge(
    dedup_updates.alias("s"),
    """
    t.user_id = s.user_id AND
    t.product_id = s.product_id AND
    t.event_time = s.event_time
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

SELECT user_id, product_id, event_time, COUNT(*)
FROM ecommerce.events_delta
GROUP BY user_id, product_id, event_time
HAVING COUNT(*) > 1;


user_id,product_id,event_time,COUNT(*)


In [0]:
v0 = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .table("ecommerce.events_delta")


In [0]:
yesterday = spark.read \
    .format("delta") \
    .option("timestampAsOf", "2024-01-01") \
    .table("ecommerce.events_delta")


In [0]:
%sql
DESCRIBE HISTORY ecommerce.events_delta;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-01-13T08:39:53.000Z,74913714958057,myuvaraj008@gmail.com,MERGE,"Map(predicate -> [""(((user_id#17133 = user_id#17110) AND (product_id#17128 = product_id#17105)) AND (event_time#17126 = event_time#17103))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3139607866975674),0113-081236-sv9h8oa1-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 93, numTargetBytesAdded -> 3742797866, numTargetBytesRemoved -> 2287249811, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 67351679, executionTimeMs -> 96444, materializeSourceTimeMs -> 37261, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 25572, numTargetRowsUpdated -> 67351679, numOutputRows -> 67351679, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 67351679, numTargetFilesRemoved -> 74, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 33504)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
2,2026-01-13T08:22:51.000Z,74913714958057,myuvaraj008@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4345491584319281),0113-081236-sv9h8oa1-v2n,1,WriteSerializable,false,"Map(numFiles -> 74, numRemovedFiles -> 85, numRemovedBytes -> 3747297381, numDeletionVectorsRemoved -> 0, numOutputRows -> 67351679, numOutputBytes -> 2287249811)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-13T08:19:07.000Z,74913714958057,myuvaraj008@gmail.com,MERGE,"Map(predicate -> [""(((user_id#13379 = user_id#13356) AND (product_id#13374 = product_id#13351)) AND (event_time#13372 = event_time#13349))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4345491584319281),0113-081236-sv9h8oa1-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 85, numTargetBytesAdded -> 3747297381, numTargetBytesRemoved -> 2574334825, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 67501979, executionTimeMs -> 110223, materializeSourceTimeMs -> 41846, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 31641, numTargetRowsUpdated -> 67501979, numOutputRows -> 67501979, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 67351679, numTargetFilesRemoved -> 23, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 36543)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-01-13T07:43:01.000Z,74913714958057,myuvaraj008@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4345491584319281),0113-071048-uzwjijxc-v2n,null,WriteSerializable,true,"Map(numFiles -> 23, numOutputRows -> 67501979, numOutputBytes -> 2574334825)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
v0 = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .table("ecommerce.events_delta")


In [0]:
v0.count()


67501979

In [0]:
s = spark.read \
    .format("delta") \
    .option("timestampAsOf", "2026-01-13 08:00:00") \
    .table("ecommerce.events_delta")


In [0]:
s.count()


67501979

In [0]:
%sql

DESCRIBE HISTORY ecommerce.events_delta;



version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-01-13T08:45:27.000Z,74913714958057,myuvaraj008@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [""event_type"",""user_id""], batchId -> 0)",null,List(3139607866975674),0113-081236-sv9h8oa1-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 93, numRemovedBytes -> 3742797866, p25FileSize -> 65624978, numDeletionVectorsRemoved -> 0, minFileSize -> 49760487, numAddedFiles -> 51, maxFileSize -> 90618716, p75FileSize -> 75684939, p50FileSize -> 69840384, numAddedBytes -> 3597074368)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
3,2026-01-13T08:39:53.000Z,74913714958057,myuvaraj008@gmail.com,MERGE,"Map(predicate -> [""(((user_id#17133 = user_id#17110) AND (product_id#17128 = product_id#17105)) AND (event_time#17126 = event_time#17103))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3139607866975674),0113-081236-sv9h8oa1-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 93, numTargetBytesAdded -> 3742797866, numTargetBytesRemoved -> 2287249811, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 67351679, executionTimeMs -> 96444, materializeSourceTimeMs -> 37261, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 25572, numTargetRowsUpdated -> 67351679, numOutputRows -> 67351679, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 67351679, numTargetFilesRemoved -> 74, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 33504)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
2,2026-01-13T08:22:51.000Z,74913714958057,myuvaraj008@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4345491584319281),0113-081236-sv9h8oa1-v2n,1,WriteSerializable,false,"Map(numFiles -> 74, numRemovedFiles -> 85, numRemovedBytes -> 3747297381, numDeletionVectorsRemoved -> 0, numOutputRows -> 67351679, numOutputBytes -> 2287249811)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-13T08:19:07.000Z,74913714958057,myuvaraj008@gmail.com,MERGE,"Map(predicate -> [""(((user_id#13379 = user_id#13356) AND (product_id#13374 = product_id#13351)) AND (event_time#13372 = event_time#13349))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4345491584319281),0113-081236-sv9h8oa1-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 85, numTargetBytesAdded -> 3747297381, numTargetBytesRemoved -> 2574334825, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 67501979, executionTimeMs -> 110223, materializeSourceTimeMs -> 41846, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 31641, numTargetRowsUpdated -> 67501979, numOutputRows -> 67501979, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 67351679, numTargetFilesRemoved -> 23, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 36543)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2026-01-13T07:43:01.000Z,74913714958057,myuvaraj008@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""

In [0]:
spark.sql("""
OPTIMIZE ecommerce.events_delta
ZORDER BY (event_type, user_id)
""")


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,